In [14]:
import pandas as pd

df = pd.read_csv("../data/gold/final_dataset.csv")
df.head()

,date,btc_close,btc_volume,fear_greed_value,value_classification,btc_daily_return,positive_return,search_interest,high_interest
0,2026-03-03,68338.00,24972.24180,14,Extreme Fear,-0.007149,0,92,1
1,2026-03-04,72666.77,44919.07610,10,Extreme Fear,0.063344,1,90,1
2,2026-03-05,70890.72,26590.51253,22,Extreme Fear,-0.024441,0,92,1
3,2026-03-06,68114.02,22629.88722,18,Extreme Fear,-0.039169,0,82,1
4,2026-03-07,67262.91,12719.54315,12,Extreme Fear,-0.012495,0,69,0


### Test 1: One-sample t-test
Is average BTC return different from 0?

In [15]:
import scipy.stats as stats

# Extract returns
returns = df['btc_daily_return']

# Run one-sample t-test (compare to 0)
t_stat, p_value = stats.ttest_1samp(returns, 0)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -0.2231782984360853
P-value: 0.8252853881088145


In [16]:
if p_value < 0.05:
    print("The average BTC return is significantly different from 0.")
else:
    print("The average BTC return is NOT significantly different from 0.")

The average BTC return is NOT significantly different from 0.


* Interpretation

The one-sample t-test was used to evaluate whether the average Bitcoin daily return is different from zero.

The p-value (0.825) is much greater than 0.05, so we fail to reject the null hypothesis.

This suggests that the average Bitcoin return in this dataset is not significantly different from zero.

* Assumptions

This test assumes that daily returns are approximately normally distributed and independent. Given the small sample size, this assumption may not fully hold.

* Limitation

The dataset covers a short time period, which may not be representative of long-term Bitcoin behavior. Results may change with a larger dataset.



### Test 2: Google Trends vs Returns
* High search interest vs Low search interest
* A paired t-test is not appropriate because the observations are not naturally paired. Instead, an independent two-sample t-test is used.

In [17]:
high = df[df['high_interest'] == 1]['btc_daily_return']
low = df[df['high_interest'] == 0]['btc_daily_return']

t_stat2, p_value2 = stats.ttest_ind(high, low)

print("T-statistic:", t_stat2)
print("P-value:", p_value2)

T-statistic: 1.6034457267236175
P-value: 0.12248182606990479


In [18]:
if p_value2 < 0.05:
    print("Returns are significantly different between high and low search interest days.")
else:
    print("No significant difference in returns between high and low search interest days.")

No significant difference in returns between high and low search interest days.


* Interpretation:

A two-sample t-test was conducted to compare average Bitcoin returns between high and low search interest days.

The p-value (0.122) is greater than 0.05, so we fail to reject the null hypothesis.

This suggests that there is no statistically significant difference in average returns between high and low search interest periods.

* Assumptions:

This test assumes that observations are independent and that returns are approximately normally distributed within each group.

* Limitation:

The sample size is relatively small, which may reduce the power of the test to detect true differences.

### Test 3: Chi-square test (categorical analysis)
* Is positive_return related to high_interest?
* Both variables are categorical, and expected frequencies are sufficient, making the chi-square test appropriate.

In [13]:
import pandas as pd
from scipy.stats import chi2_contingency

table = pd.crosstab(df['positive_return'], df['high_interest'])

chi2, p, dof, expected = chi2_contingency(table)

print("P-value:", p)

P-value: 0.5532154726633813


* Interpretation:

A chi-square test of independence was performed to examine whether positive returns are associated with high search interest.

The p-value (0.55) is greater than 0.05, so we fail to reject the null hypothesis.

This suggests that the likelihood of positive returns is independent of search interest levels.

* Assumptions:

This test assumes that observations are independent and that expected cell counts are sufficiently large.

* Limitation:

The dataset size may limit the reliability of categorical comparisons.

### Test 4: Variance comparison
* Is volatility (variance of returns) different between high vs low interest?
* Levene’s test is used instead of a classical F-test because it does not assume normality and is more appropriate for financial return data.

In [19]:
high = df[df['high_interest'] == 1]['btc_daily_return']
low = df[df['high_interest'] == 0]['btc_daily_return']

var_high = high.var()
var_low = low.var()

print("Variance (High interest):", var_high)
print("Variance (Low interest):", var_low)

Variance (High interest): 0.001072006602856809
Variance (Low interest): 0.00032380081827542844


In [20]:
from scipy.stats import levene

stat, p_value = levene(high, low)

print("Levene test p-value:", p_value)

Levene test p-value: 0.04531134915170209


* Interpretation:

A variance comparison using Levene’s test was conducted to evaluate whether return volatility differs between high and low search interest days.

The p-value (0.045) is less than 0.05, so we reject the null hypothesis of equal variances.

This indicates that Bitcoin returns are significantly more volatile during high search interest periods.

* Assumptions:

Levene’s test does not assume normality and is appropriate for comparing variances across groups.

* Limitation:

While variance differs, this does not imply causation, and other external factors may influence volatility.

### Test 5: Correlation analysis
* Is search_interest correlated with BTC returns?
* Pearson correlation is used to measure the linear relationship between search interest and returns. However, results may be sensitive to outliers and non-linear patterns.

In [21]:
from scipy.stats import pearsonr

corr, p_value = pearsonr(df['search_interest'], df['btc_daily_return'])

print("Correlation:", corr)
print("P-value:", p_value)

Correlation: 0.30270840905397167
P-value: 0.14134036758926757


* Interpretation:

A Pearson correlation test was used to examine the relationship between search interest and Bitcoin returns.

The correlation coefficient (0.30) suggests a weak positive relationship.

However, the p-value (0.14) is greater than 0.05, indicating that this relationship is not statistically significant.

* Assumptions:

Pearson correlation assumes a linear relationship and is sensitive to outliers.

* Limitation:

The lack of statistical significance may be due to limited sample size or variability in the data.

### Technical Justification

The statistical methods used in this analysis were selected based on the structure of the data and the research questions.

A paired t-test was not appropriate because the observations are not naturally paired; therefore, an independent two-sample t-test was used.

The chi-square test was suitable because both variables (positive return and search interest category) are categorical with sufficient expected counts.

For variance comparison, Levene’s test was chosen instead of an F-test because it is more robust to non-normal distributions, which are common in financial return data.

Pearson correlation was used to assess the linear relationship between search interest and returns, although the results may be affected by outliers or non-linear patterns.